# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step demonstration for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is published with a [Croissant schema metadata file](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

> **Note:** All entities in the dataset (record sets, fields, etc) are referenced by their `@id` for full reproducibility and clarity.


In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant --quiet

## 1. Data Loading
We load the dataset and schema metadata using `mlcroissant`. The main dataset is described by the Croissant-JSON-LD schema URL below.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's list available record sets defined in this dataset and inspect the fields per record set, referencing all by their `@id`.

In [ ]:
# List all record sets and their fields by `@id`
record_sets = metadata.record_sets # This returns a list of RecordSet objects
print(f'Total RecordSets: {len(record_sets)}\n')
rs_ids = []

for rs in record_sets:
    print(f"RecordSet name: {rs.name} (@id: {rs.id})\n  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, DataType: {field.data_type})")
    print()
    rs_ids.append(rs.id)

# We'll use the first record set as an example for analysis below
primary_recordset_id = rs_ids[0] if rs_ids else None

print(f"Primary record set for analysis: {primary_recordset_id}")

## 3. Data Extraction
Load all data from the chosen record set (by `@id`) into a Pandas DataFrame. We'll loop over all available record sets and build a DataFrame for each, referenced by their `@id`.


In [ ]:
# Extract all records for each record set, loading them into DataFrames.
dataframes = {}

for rs in record_sets:
    print(f"Loading records for RecordSet: {rs.name} (@id={rs.id})...")
    try:
        records = list(dataset.records(record_set=rs.id))
        if records: # records is a list of dict
            df = pd.DataFrame(records)
            dataframes[rs.id] = df
            print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
            print(df.head(2), "\n")
        else:
            print("No records found.")
    except Exception as e:
        print(f"Failed to load: {e}")


## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA on the primary record set (by `@id`). We will:

- Select a numeric field (e.g., 'MW' or 'Coverage (%)') by field `@id`
- Filter records for values above a threshold
- Normalize the numeric values
- Optionally group by another field (e.g., a sample name or annotation)


In [ ]:
import numpy as np

# Select a numeric field in the primary record set, referenced by its Croissant field @id
numeric_field_id = None
group_field_id = None

# Pick first numeric field if available
df = dataframes.get(primary_recordset_id)
if df is not None:
    # Find a numeric column (float or int), usually with 'MW', 'Coverage', 'Peptides', etc
    for f in [f for f in metadata.record_sets[0].fields]:
        # Try numeric types
        if f.data_type in ['Float', 'Integer', 'Number']:
            if f.id in df.columns:
                numeric_field_id = f.id
                break
    # For grouping, pick first Text field
    for f in [f for f in metadata.record_sets[0].fields]:
        if f.data_type == 'Text' and f.id in df.columns:
            group_field_id = f.id
            break
    print(f"Using numeric field '{numeric_field_id}' and group field '{group_field_id}' for EDA.\n")

    # Filter records for numeric_field > threshold (e.g. mean)
    threshold = df[numeric_field_id].mean() if numeric_field_id else None
    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df.copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} of {len(df)}\n")
    
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped.head())
else:
    print("No data found for the primary record set.")

## 5. Visualization
Plot distributions of the selected numeric field and grouping, using Matplotlib.

> The plot functions below will render only if the DataFrame extraction above was successful.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot per group
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('Cannot plot: dataset not loaded, or missing numeric field.')

## 6. Conclusion
- The dataset contains data on protein abundance and modifications in human samples as isolated via mass spectrometry.
- The Croissant schema makes it easy to identify record sets and their fields by `@id`, enabling robust programmatic analysis.
- We performed basic filtering, normalization, and grouped aggregation—and visualized the main quantitative field.

To go further: try extracting other record sets, exploring text fields, or merging with external ontologies or databases using the Croissant `@id` structure.
